In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
df = pd.read_csv("train.csv")
df = df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Convert categorical variables to numbers
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# Separate features and target
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
original_model = LogisticRegression(max_iter=1000)
original_model.fit(X_train, y_train)
# Predictions
y_pred_original = original_model.predict(X_test)

# Performance
original_accuracy = accuracy_score(y_test, y_pred_original)

print("Original Model Performance:")
print(classification_report(y_test, y_pred_original))
# Hyperparameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

# Grid Search
grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='f1'
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)
# Best/tuned model
tuned_model = grid_search.best_estimator_

# Predictions
y_pred_tuned = tuned_model.predict(X_test)

# Performance
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

print("Tuned Model Performance:")
print(classification_report(y_test, y_pred_tuned))

Original Model Performance:
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

Best Parameters:
{'C': 0.1, 'solver': 'lbfgs'}
Tuned Model Performance:
              precision    recall  f1-score   support

           0       0.80      0.90      0.85       110
           1       0.80      0.64      0.71        69

    accuracy                           0.80       179
   macro avg       0.80      0.77      0.78       179
weighted avg       0.80      0.80      0.79       179



In [2]:
#COMPARISION
original_precision = precision_score(y_test, y_pred_original)
original_recall = recall_score(y_test, y_pred_original)
original_f1 = f1_score(y_test, y_pred_original)

tuned_precision = precision_score(y_test, y_pred_tuned)
tuned_recall = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Original Model': [
        original_accuracy,
        original_precision,
        original_recall,
        original_f1
    ],
    'Tuned Model': [
        tuned_accuracy,
        tuned_precision,
        tuned_recall,
        tuned_f1
    ]
})

print(comparison)

      Metric  Original Model  Tuned Model
0   Accuracy        0.804469     0.798883
1  Precision        0.793103     0.800000
2     Recall        0.666667     0.637681
3   F1-Score        0.724409     0.709677


**Why accuracy alone can be misleading for imbalanced datasets?**
Accuracy alone can be misleading when the dataset is imbalanced because it only tells us the overall percentage of correct predictions. For example, if 90% of the data belongs to one class, a model that always predicts that class could achieve 90% accuracy while completely failing to identify the minority class. Precision, recall, and F1-score give us a better understanding of how well the model performs for each class.